In [240]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [241]:
result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/ours/combined_stage_label_with_evidence_and_mlp_06_11_2025.pkl")

In [242]:
result_df.shape

(3520, 12)

In [6]:
import unicodedata
from typing import Iterable, List, Literal

KeepRule = Literal["letters", "alnum"]

def clean_token(
    s: object,
    keep: KeepRule = "letters",
    strip_accents: bool = False,
) -> str:
    """
    Normalize a single token:
      - converts to str, NFKC normalizes, lowercases
      - optionally removes accents
      - keeps only letters (default) or letters+digits
    Returns '' if nothing remains.

    Examples:
      "  Dogs," -> "dogs"
      "Car." -> "car"
      "V2.0!" (alnum) -> "v20"
    """
    if s is None:
        return ""
    # Normalize width/compat (e.g., full-width chars), lowercase
    t = unicodedata.normalize("NFKC", str(s)).lower().strip()

    # Optional: remove diacritics (é -> e)
    if strip_accents:
        t = "".join(
            ch for ch in unicodedata.normalize("NFKD", t)
            if unicodedata.category(ch) != "Mn"
        )

    if keep == "letters":
        # Keep only unicode letters
        return "".join(ch for ch in t if ch.isalpha())
    elif keep == "alnum":
        # Keep letters and digits (useful if your tokens may include numbers)
        return "".join(ch for ch in t if ch.isalnum())
    else:
        raise ValueError("keep must be 'letters' or 'alnum'")

def clean_tokens(
    tokens: Iterable[object],
    keep: KeepRule = "letters",
    strip_accents: bool = False,
) -> List[str]:
    """
    Clean a sequence of tokens and drop empties.
    """
    out = []
    for tok in tokens:
        ct = clean_token(tok, keep=keep, strip_accents=strip_accents)
        if ct:
            out.append(ct)
    return out

In [26]:
import re

def word_membership_vector(sentence, targets):
    """
    Returns a list of 1/0 indicating whether each word in `sentence`
    (lowercased, punctuation stripped) is in `targets` (also lowercased).
    """
    words = re.findall(r"\w+", sentence.lower())
    target_set = {t.lower() for t in targets}
    return [1 if w in target_set else 0 for w in words]

In [69]:

all_precisions = []
all_recalls = []
all_f1s = []

all_tps = []
all_fps = []
all_fns = []


all_predictions = []
for idx, row in good_df_1.iloc[0:].iterrows():
    target_words = eval(row['hallucination_candidates'])
    res = row["labels_with_evidence"]
    hal_words = []
    cand_words = [] 
    for i in res:
        viz_evi = (torch.tensor(i["evidence"]) >= 0.5).int().sum().item()
        prob = i["label"]
        if prob <= 0.6:
            hal_words.append(i["word"])
        elif viz_evi <= 3 and prob <= 0.95:
            hal_words.append(i["word"])
        
        cand_words.append(i["word"])
    
            
    hal_words = clean_tokens(tokens = hal_words, keep = "alnum", strip_accents= True,)
    all_predictions.append(hal_words)
    target_words = clean_tokens(tokens = target_words, keep = "alnum", strip_accents= True,)

    tps = set(hal_words).intersection(set(target_words))
    all_tps.extend(tps)
    fps = set(hal_words) - set(tps)
    all_fps.extend(fps)
    fns = set(target_words) - set(tps)
    all_fns.extend(fns)
    precision = len(tps) / (len(tps) + len(fps) + 1e-8)
    recall = len(tps) / (len(tps) + len(fns) + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    all_precisions.append(precision)
    all_recalls.append(recall)
    all_f1s.append(f1)

In [206]:
good_df_1 = result_df[(result_df["f1"] >= 0.57) | (result_df["hallucination_candidates"].apply(lambda x: len(eval(x)) <= 2) & result_df["predictions"].apply(lambda x: len(x) <= 3))]

In [207]:
good_df_1.shape

(3520, 12)

In [218]:
len(all_halu_detection_gts)

36917

In [247]:

def get_all_metrics(target_df):

    all_candidate_selection_preds = []
    all_candidate_selection_gts = []

    all_halu_detection_preds = []
    all_halu_detection_gts = []

    num_samples = 0
    for idx, row in target_df.iloc[0:].iterrows():
        gt_halu_words = eval(row['hallucination_candidates'])
        gt_non_halu_words = eval(row['candidates'])
        answer = row['answer']
        res = row["labels_with_evidence"]
        pred_hal_words = []
        pred_non_hal_words = [] 
        for i in res:
            viz_evi = (torch.tensor(i["evidence"]) >= 0.5).int().sum().item()
            prob = i["label"]
            if prob <= 0.75:
                pred_hal_words.append(i["word"])
            elif viz_evi <= 5 and prob <= 0.95:
                pred_hal_words.append(i["word"])
            else:
                pred_non_hal_words.append(i["word"])

        
        gt_halu_words = clean_tokens(tokens = gt_halu_words, keep = "alnum", strip_accents= True,)
        gt_non_halu_words = clean_tokens(tokens = gt_non_halu_words, keep = "alnum", strip_accents= True,)
        pred_hal_words = clean_tokens(tokens = pred_hal_words, keep = "alnum", strip_accents= True,)
        pred_non_hal_words = clean_tokens(tokens = pred_non_hal_words, keep = "alnum", strip_accents= True,)
        
        ## candidate slection performance
        gt_candidate_selected = list(set(gt_halu_words + gt_non_halu_words))
        pred_candidate_selected = list(set(pred_hal_words + pred_non_hal_words))
        
        gt_candidate_labels = word_membership_vector(answer, gt_candidate_selected)
        pred_candidate_labels = word_membership_vector(answer, pred_candidate_selected)
        
        all_candidate_selection_preds.extend(pred_candidate_labels)
        all_candidate_selection_gts.extend(gt_candidate_labels)
        
        gt_halu_labels = []
        pred_halu_labels = []
        for token in gt_candidate_selected:
            if token in gt_halu_words:
                gt_halu_labels.append(1)
            elif token in gt_non_halu_words:
                gt_halu_labels.append(0)
            else:
                raise ValueError("Token not found in ground truth candidates")
            
            if token in pred_hal_words:
                pred_halu_labels.append(1)
            elif token in pred_non_hal_words:
                pred_halu_labels.append(0)
            else:
                num_samples += 1
                pred_halu_labels.append(0)
        
        all_halu_detection_preds.extend(pred_halu_labels)
        all_halu_detection_gts.extend(gt_halu_labels)
    
    
    candidate_selection_accuracy = accuracy_score(all_candidate_selection_gts, all_candidate_selection_preds)
    candidate_selection_precision = precision_score(all_candidate_selection_gts, all_candidate_selection_preds)
    candidate_selection_recall = recall_score(all_candidate_selection_gts, all_candidate_selection_preds)
    candidate_selection_f1 = f1_score(all_candidate_selection_gts, all_candidate_selection_preds)
    
    halu_detection_accuracy = accuracy_score(all_halu_detection_gts, all_halu_detection_preds)
    halu_detection_precision = precision_score(all_halu_detection_gts, all_halu_detection_preds)
    halu_detection_recall = recall_score(all_halu_detection_gts, all_halu_detection_preds)
    halu_detection_f1 = f1_score(all_halu_detection_gts, all_halu_detection_preds)
    return {
        "candidate_selection_accuracy": candidate_selection_accuracy,
        "candidate_selection_precision": candidate_selection_precision,
        "candidate_selection_recall": candidate_selection_recall,
        "candidate_selection_f1": candidate_selection_f1,
        "halu_detection_accuracy": halu_detection_accuracy,
        "halu_detection_precision": halu_detection_precision,
        "halu_detection_recall": halu_detection_recall,
        "halu_detection_f1": halu_detection_f1,
    }

In [250]:
result_df["data_type"].value_counts()

data_type
vqa_nonhalu_llava         472
vqa_nonhalu_qwen          469
vqa_halu_llava            417
instruct_nonhalu_llava    416
instruct_nonhalu_qwen     363
instruct_halu_llava       358
vqa_halu_qwen             279
caption_nonhalu_llava     278
caption_nonhalu_qwen      265
instruct_halu_qwen        203
Name: count, dtype: int64

In [258]:
target_df = result_df[result_df["data_type"].isin(["caption_nonhalu_llava", "caption_nonhalu_qwen"])] 
get_all_metrics(target_df)

{'candidate_selection_accuracy': 0.9259963464191179,
 'candidate_selection_precision': 0.9599351330725937,
 'candidate_selection_recall': 0.8654110767113863,
 'candidate_selection_f1': 0.9102256795260277,
 'halu_detection_accuracy': 0.8661790867760427,
 'halu_detection_precision': 0.6997624703087886,
 'halu_detection_recall': 0.6710706150341685,
 'halu_detection_f1': 0.6851162790697675}